<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/EarningsLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install scipy==1.16.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: yfinance
    Found existing installation: yfinance 0.2.66
    Uninstalling yfinance-0.2.66:
      Successfully uninstalled yfinance-0.2.66
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 47.8 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from math import sqrt
print("Libraries installed successfully!")

Libraries installed successfully!


## Build earnings lab class

In [36]:

def find_next_trading_day(price_data, target_date, max_days=10):
    trading_days = sorted(price_data.index)
    for d in trading_days:
        if d >= target_date:
            return d
    return None

def earnings_edge_engine(ticker_symbol, lookback=16):
    print(f"\nAnalyzing {ticker_symbol}...")

    ticker = yf.Ticker(ticker_symbol)

    # ====================== 1. EARNINGS DATES ======================
    earnings = ticker.get_earnings_dates(limit=lookback)
    if earnings is None or earnings.empty:
        print(f"No earnings data for {ticker_symbol}")
        return None

    earnings = earnings.reset_index()
    earnings['Earnings Date'] = pd.to_datetime(earnings['Earnings Date']).dt.tz_localize(None)
    today = pd.Timestamp.today().normalize()

    past_earnings = earnings[earnings['Earnings Date'] < today]
    future_earnings = earnings[earnings['Earnings Date'] >= today]

    if future_earnings.empty:
        print(f"No upcoming earnings for {ticker_symbol}")
        return None

    next_earnings_date = future_earnings.iloc[0]['Earnings Date'].date()

    # ====================== 2. PRICE DATA & HISTORICAL MOVES ======================
    price_data = ticker.history(period="5y", auto_adjust=True)
    price_data.index = price_data.index.date

    results = []

    for _, row in past_earnings.iterrows():
        announce_date = row['Earnings Date'].date()
        event_day = find_next_trading_day(price_data, announce_date)
        if not event_day:
            continue

        prior_day = find_next_trading_day(price_data, announce_date - timedelta(days=7))
        if not prior_day or prior_day >= event_day:
            continue

        try:
            prior_close = price_data.loc[prior_day]['Close']
            event_day_data = price_data.loc[event_day]
            next_day = find_next_trading_day(price_data, event_day + timedelta(days=1))
            if not next_day:
                continue
            next_close = price_data.loc[next_day]['Close']

            two_day_pct = (next_close - prior_close) / prior_close * 100

            results.append({
                "Earnings Date": announce_date,
                "Event Day": event_day,
                "Post Earnings Move %": round(abs(two_day_pct), 2),
                "Direction": "Up" if two_day_pct > 0 else "Down"
            })
        except:
            continue

    if not results:
        print(f"No valid historical moves for {ticker_symbol}")
        return None

    df_moves = pd.DataFrame(results)

    hist_median = df_moves["Post Earnings Move %"].median()
    hist_mean = df_moves["Post Earnings Move %"].mean()
    hist_std = df_moves["Post Earnings Move %"].std()
    top3 = df_moves.nlargest(3, 'Post Earnings Move %')[['Earnings Date', 'Post Earnings Move %', 'Direction']]

    # ====================== 3. OPTIONS + IV RANK ======================
    expirations = ticker.options
    if not expirations:
        print(f"No options for {ticker_symbol}")
        return None

    exp_dates = [pd.to_datetime(e).date() for e in expirations]
    valid_exps = [e for e in exp_dates if e > next_earnings_date]
    if not valid_exps:
        print(f"No valid expiration for {ticker_symbol}")
        return None

    target_exp = min(valid_exps)
    option_chain = ticker.option_chain(str(target_exp))

    calls = option_chain.calls
    puts = option_chain.puts
    current_price = price_data.iloc[-1]['Close']

    calls["distance"] = abs(calls["strike"] - current_price)
    puts["distance"] = abs(puts["strike"] - current_price)

    atm_call = calls.loc[calls["distance"].idxmin()]
    atm_put = puts.loc[puts["distance"].idxmin()]

    # Mid price with liquidity check
    call_mid = (atm_call['bid'] + atm_call['ask']) / 2
    put_mid = (atm_put['bid'] + atm_put['ask']) / 2
    straddle_price = call_mid + put_mid

    implied_move_pct = (straddle_price / current_price) * 100

    # Current ATM IV
    current_iv = (atm_call['impliedVolatility'] + atm_put['impliedVolatility']) / 2 * 100

    # Simple IV Rank (approximation using 1 year history)
    try:
        hist_vol = ticker.history(period="1y")['Close'].pct_change().std() * np.sqrt(252) * 100
        iv_rank = min(max((current_iv - 20) / (hist_vol * 1.5), 0), 100)  # Rough but useful approximation
    except:
        iv_rank = None

    # ====================== 4. BIAS LOGIC (Improved) ======================
    ratio = implied_move_pct / hist_median if hist_median > 0 else 1.0

    if ratio > 1.5 and (iv_rank is None or iv_rank > 60):
        bias = "STRONG SELL VOLATILITY"
        suggested_structure = "Short Iron Condor"
    elif ratio > 1.3:
        bias = "SELL VOLATILITY"
        suggested_structure = "Short Iron Condor"
    elif ratio < 0.6 and (iv_rank is None or iv_rank < 40):
        bias = "STRONG BUY VOLATILITY"
        suggested_structure = "Long Straddle / Strangle"
    elif ratio < 0.80:
        bias = "BUY VOLATILITY"
        suggested_structure = "Long Straddle"
    else:
        bias = "NEUTRAL / FAIR"
        suggested_structure = "Directional or Calendar Spread"

    bias_score = round((ratio - 1) * 100, 2)
    up_moves = df_moves[df_moves["Direction"] == "Up"].shape[0]
    down_moves = df_moves[df_moves["Direction"] == "Down"].shape[0]

    directional_bias = "Bullish" if up_moves > down_moves else "Bearish"

    # ====================== 5. FINAL OUTPUT ======================
    print(f"===== {ticker_symbol} Earnings Edge Analysis =====")
    print(f"Next Earnings      : {next_earnings_date}")
    print(f"Current Price       : ${round(current_price, 2)}")
    print(f"Implied Move        : {round(implied_move_pct, 2)}%")
    print(f"Historical Median   : {round(hist_median, 2)}%")
    print(f"Historical Mean     : {round(hist_mean, 2)}%")
    print(f"Ratio (Imp/Median)  : {round(ratio, 2)}x")
    if iv_rank is not None:
        print(f"IV Rank             : {round(iv_rank, 1)}%")
    print(f"Directional bias is    : {directional_bias}")
    print(f"Recommendation      : {bias}")
    print(f"Suggested Structure : {suggested_structure}\n")

    summary_df = pd.DataFrame([{
        "Ticker": ticker_symbol,
        "Next Earnings": next_earnings_date,
        "Current Price": round(current_price, 2),
        "Implied Move %": round(implied_move_pct, 2),
        "Hist Median %": round(hist_median, 2),
        "Ratio": round(ratio, 2),
        "IV Rank %": round(iv_rank, 1) if iv_rank is not None else None,
        "Bias Score": bias_score,
        "Direction bias": directional_bias,
        "Bias": bias,
        "Suggested Structure": suggested_structure,
        "Top 3 Moves": top3.to_dict(orient='records')
    }])

    return summary_df


def earnings_edge_batch(tickers, lookback=16):
    if isinstance(tickers, str):
        tickers = [tickers]

    all_results = []
    for t in tickers:
        try:
            result = earnings_edge_engine(t, lookback)
            if result is not None:
                all_results.append(result)
        except Exception as e:
            print(f"Error on {t}: {e}")

    if all_results:
        df = pd.concat(all_results, ignore_index=True)
        print(f"\n=== FINAL SUMMARY TABLE ===")
        print(df[['Ticker', 'Implied Move %', 'Hist Median %', 'Ratio', 'IV Rank %', 'Bias']].sort_values(by='Ratio', ascending=False))
        return df
    return pd.DataFrame()

In [37]:
my_list = ["UNH", "GE", "COF", "RTX", "TSLA", "BA", "NOW", "LRCX", "TXN", "IBM","VRT", "GEV", "INTC", "FCX", "NEM", "UNP",
           "BX", "AXP","NEE", "LMT", "SLB", "PG"]
df_results = earnings_edge_batch(my_list)
df_results
#df = earnings_edge_engine(my_list)
#df


Analyzing UNH...
===== UNH Earnings Edge Analysis =====
Next Earnings      : 2026-04-21
Current Price       : $323.48
Implied Move        : 5.98%
Historical Median   : 2.77%
Historical Mean     : 5.46%
Ratio (Imp/Median)  : 2.16x
IV Rank             : 0.7%
Directional bias is    : Bearish
Recommendation      : SELL VOLATILITY
Suggested Structure : Short Iron Condor


Analyzing GE...
===== GE Earnings Edge Analysis =====
Next Earnings      : 2026-04-21
Current Price       : $303.6
Implied Move        : 5.13%
Historical Median   : 3.34%
Historical Mean     : 4.47%
Ratio (Imp/Median)  : 1.53x
IV Rank             : 0.9%
Directional bias is    : Bullish
Recommendation      : SELL VOLATILITY
Suggested Structure : Short Iron Condor


Analyzing COF...
===== COF Earnings Edge Analysis =====
Next Earnings      : 2026-04-21
Current Price       : $205.71
Implied Move        : 5.18%
Historical Median   : 6.6%
Historical Mean     : 6.14%
Ratio (Imp/Median)  : 0.79x
IV Rank             : 0.9%
Direct

,Ticker,Next Earnings,Current Price,Implied Move %,Hist Median %,Ratio,IV Rank %,Bias Score,Direction bias,Bias,Suggested Structure,Top 3 Moves
0,UNH,2026-04-21,323.48,5.98,2.77,2.16,0.7,115.95,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-17, 'Post Earnings ..."
1,GE,2026-04-21,303.60,5.13,3.34,1.53,0.9,53.37,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-26, 'Post Earnings ..."
2,COF,2026-04-21,205.71,5.18,6.60,0.79,0.9,-21.50,Bullish,BUY VOLATILITY,Long Straddle,"[{'Earnings Date': 2022-10-27, 'Post Earnings ..."
3,RTX,2026-04-21,195.79,5.27,4.22,1.25,1.1,24.82,Bullish,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2025-10-21, 'Post Earnings ..."
4,TSLA,2026-04-22,392.50,5.93,5.19,1.14,0.7,14.26,Bearish,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2023-01-25, 'Post Earnings ..."
5,BA,2026-04-22,225.08,4.90,2.01,2.44,0.8,143.69,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-27, 'Post Earnings ..."
6,NOW,2026-04-22,99.72,10.23,4.46,2.29,1.6,129.34,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-23, 'Post Earnings ..."
7,LRCX,2026-04-22,263.16,7.75,5.92,1.31,1.0,30.94,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2024-01-24, 'Post Earnings ..."
8,TXN,2026-04-22,233.70,6.18,4.19,1.48,1.0,47.75,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-07-22, 'Post Earnings ..."
9,IBM,2026-04-22,253.71,6.79,4.24,1.60,1.3,60.12,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-01-29, 'Post Earnings ..."


In [38]:
# Sort by richest volatility (best for selling premium)
df_results.sort_values(by="Ratio", ascending=False)

,Ticker,Next Earnings,Current Price,Implied Move %,Hist Median %,Ratio,IV Rank %,Bias Score,Direction bias,Bias,Suggested Structure,Top 3 Moves
5,BA,2026-04-22,225.08,4.90,2.01,2.44,0.8,143.69,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-27, 'Post Earnings ..."
6,NOW,2026-04-22,99.72,10.23,4.46,2.29,1.6,129.34,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-23, 'Post Earnings ..."
0,UNH,2026-04-21,323.48,5.98,2.77,2.16,0.7,115.95,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-17, 'Post Earnings ..."
9,IBM,2026-04-22,253.71,6.79,4.24,1.60,1.3,60.12,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-01-29, 'Post Earnings ..."
21,PG,2026-04-24,144.49,4.42,2.78,1.59,0.5,59.08,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-24, 'Post Earnings ..."
13,FCX,2026-04-23,70.18,5.52,3.49,1.58,0.7,58.44,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-01-26, 'Post Earnings ..."
1,GE,2026-04-21,303.60,5.13,3.34,1.53,0.9,53.37,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-26, 'Post Earnings ..."
8,TXN,2026-04-22,233.70,6.18,4.19,1.48,1.0,47.75,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-07-22, 'Post Earnings ..."
12,INTC,2026-04-23,65.70,10.12,7.62,1.33,1.1,32.92,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2024-08-01, 'Post Earnings ..."
7,LRCX,2026-04-22,263.16,7.75,5.92,1.31,1.0,30.94,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2024-01-24, 'Post Earnings ..."


In [39]:
# Or see only the ones worth trading
df_results[df_results["Ratio"] > 1.25]   # Good for selling vol

,Ticker,Next Earnings,Current Price,Implied Move %,Hist Median %,Ratio,IV Rank %,Bias Score,Direction bias,Bias,Suggested Structure,Top 3 Moves
0,UNH,2026-04-21,323.48,5.98,2.77,2.16,0.7,115.95,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-17, 'Post Earnings ..."
1,GE,2026-04-21,303.60,5.13,3.34,1.53,0.9,53.37,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-26, 'Post Earnings ..."
5,BA,2026-04-22,225.08,4.90,2.01,2.44,0.8,143.69,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-04-27, 'Post Earnings ..."
6,NOW,2026-04-22,99.72,10.23,4.46,2.29,1.6,129.34,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-04-23, 'Post Earnings ..."
7,LRCX,2026-04-22,263.16,7.75,5.92,1.31,1.0,30.94,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2024-01-24, 'Post Earnings ..."
8,TXN,2026-04-22,233.70,6.18,4.19,1.48,1.0,47.75,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-07-22, 'Post Earnings ..."
9,IBM,2026-04-22,253.71,6.79,4.24,1.60,1.3,60.12,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2025-01-29, 'Post Earnings ..."
12,INTC,2026-04-23,65.70,10.12,7.62,1.33,1.1,32.92,Bearish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2024-08-01, 'Post Earnings ..."
13,FCX,2026-04-23,70.18,5.52,3.49,1.58,0.7,58.44,Bullish,SELL VOLATILITY,Short Iron Condor,"[{'Earnings Date': 2022-01-26, 'Post Earnings ..."
14,NEM,2026-04-23,114.84,6.68,5.32,1.26,0.9,25.62,Bearish,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2022-07-25, 'Post Earnings ..."
